### 2 Week 7 [Feb 10 - Feb 16]: Collaborative Filtering Recommender System

1. Based on the frequency of the most rated items computed in Week 6, implement the TopPop recommender system, which always recommends
the same top-k items sorted decreasingly by the number of “high” ratings (e.g., ≥ 3) in the training split, train.tsv.

In [53]:
import pandas as pd

class TopPop:
    def __init__(self, k=10):
        self.k = k
        self.top_items = None

    def fit(self, file_path):
        # Read the training data (no header assumed)
        df = pd.read_csv(file_path, sep='\t')
        # Keep only the high ratings (e.g., >= 3)
        df_high = df[df['rating'] >= 3]
        # Compute the frequency of high ratings per item and sort in decreasing order
        counts = df_high['item_id'].value_counts()
        self.top_items = list(counts.index)

    def recommend(self, user_id):
        # Return the top-k items (this model is user-agnostic)
        return self.top_items[:self.k]
    
top_pop = TopPop(k=10)
top_pop.fit('Data/clean_train.tsv')

2. Choose at least one neighborhood-based model and one latent factor model that uses the observed user-item ratings in the training set to predict the unobserved ratings. Report your choice of models.

In [54]:
from surprise import Dataset, Reader, KNNWithMeans, SVD
import pandas as pd

# For the neighborhood-based model, I chose the KNNWithMeans collaborative filtering algorithm.
# In particular, the KNNWithMeans implementation from the surprise library is a simple yet effective approach 
# that computes similarities between users (or items) and makes predictions based on the ratings from 
# the most similar neighbors.
#
# For the latent factor model, I selected the Singular Value Decomposition (SVD) algorithm.
# SVD decomposes the user-item rating matrix into latent factors, capturing the underlying associations 
# between users and items. This model has been widely used in collaborative filtering tasks with excellent performance.
#
# Below is an example to illustrate how you might set up and train these models
# using the surprise package:

# Load your training data from the file (assuming train.tsv is in 'Data/clean_train.tsv')
df = pd.read_csv('Data/clean_train.tsv', sep='\t')
reader = Reader(rating_scale=(1, 5))

# Prepare the dataset using the surprise framework
data = Dataset.load_from_df(df[['user_id', 'item_id', 'rating']], reader)
trainset = data.build_full_trainset()

3. Use 5-fold cross-validation on the training set to tune the hyperparameters of the chosen models (similarity measure and number of neighbors for the neighborhood-based model; number of latent factors and number of epochs for the latent factor model).

In [55]:
from surprise.model_selection import GridSearchCV

# Define parameter grid for KNNWithMeans (neighborhood-based model)
param_grid_knn = {
    'k': [5, 10, 20],
    'verbose': [False],
    'sim_options': {
        'name': ['cosine', 'msd'],
        'user_based': [True]  # We use user-based collaborative filtering
    }
}

# Define parameter grid for SVD (latent factor model)
param_grid_svd = {
    'n_factors': [20, 50, 100],
    'n_epochs': [20, 50, 100]
}

# Perform 5-fold cross-validation for KNNWithMeans
grid_knn = GridSearchCV(KNNWithMeans, param_grid_knn, measures=['rmse', 'mae'], cv=5)
grid_knn.fit(data)

print("Best KNN parameters (based on RMSE):", grid_knn.best_params['rmse'])
print("Best RMSE for KNN:", grid_knn.best_score['rmse'])

# Perform 5-fold cross-validation for SVD
grid_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=5)
grid_svd.fit(data)

Best KNN parameters (based on RMSE): {'k': 20, 'verbose': False, 'sim_options': {'name': 'cosine', 'user_based': True}}
Best RMSE for KNN: 0.9078691270132726


4. Choose an evaluation measure that is suitable for this task and justify your motivation in using it. Report the optimal hyperparameters together with the scores of your chosen measure, averaged over the 5 folds.

In [56]:
# We choose RMSE (Root Mean Squared Error) as the evaluation measure.
# RMSE penalizes larger errors more than smaller ones, making it a good measure to
# capture the overall predictive accuracy of our recommendation models.
# Lower RMSE indicates that the model predictions are closer to the actual ratings.

print("\nKNNWithMeans Best Hyperparameters (based on RMSE):")
print(grid_knn.best_params['rmse'])
print("Average RMSE for KNNWithMeans:", grid_knn.best_score['rmse'])

print("\nSVD Best Hyperparameters (based on RMSE):")
print(grid_svd.best_params['rmse'])
print("Average RMSE for SVD:", grid_svd.best_score['rmse'])


KNNWithMeans Best Hyperparameters (based on RMSE):
{'k': 20, 'verbose': False, 'sim_options': {'name': 'cosine', 'user_based': True}}
Average RMSE for KNNWithMeans: 0.9078691270132726

SVD Best Hyperparameters (based on RMSE):
{'n_factors': 20, 'n_epochs': 20}
Average RMSE for SVD: 0.8442067408444209


5. Run the models with the optimal hyperparameters to the whole training set.

In [57]:
# Re-train the models on the full training set using the optimal hyperparameters from the grid searches

# For KNNWithMeans
knn_best = grid_knn.best_estimator['rmse']
knn_best.fit(trainset)

# For SVD
svd_best = grid_svd.best_estimator['rmse']
svd_best.fit(trainset)

print("KNNWithMeans model trained with optimal hyperparameters.")
print("SVD model trained with optimal hyperparameters.")

KNNWithMeans model trained with optimal hyperparameters.
SVD model trained with optimal hyperparameters.


6. Use the final models to rank the non-rated items for each user. This ranking will be used for the evaluation part next week.

In [58]:
from collections import defaultdict

# Build the anti-testset (all user-item pairs not in the training set)
anti_testset = trainset.build_anti_testset()

# Get predictions on the anti-testset using the KNNWithMeans model
predictions = knn_best.test(anti_testset)

# Group predictions by user
user_recs = defaultdict(list)
for pred in predictions:
    user_recs[pred.uid].append((pred.iid, pred.est))

# For each user, sort the recommendations in descending order of predicted rating
for user in user_recs:
    user_recs[user] = [iid for iid, _ in sorted(user_recs[user], key=lambda x: x[1], reverse=True)]

# Display sample recommendations for a few users
print("Sample recommendations using KNNWithMeans:")
for user, recs in list(user_recs.items())[:5]:
    print(f"User {user}: {recs[:10]}")

# Get predictions on the anti-testset using the SVD model
predictions = svd_best.test(anti_testset)

# Group predictions by user
user_recs = defaultdict(list)
for pred in predictions:
    user_recs[pred.uid].append((pred.iid, pred.est))

# For each user, sort the recommendations in descending order of predicted rating
for user in user_recs:
    user_recs[user] = [iid for iid, _ in sorted(user_recs[user], key=lambda x: x[1], reverse=True)]

# Display sample recommendations for a few users
print("Sample recommendations using SVD:")
for user, recs in list(user_recs.items())[:5]:
    print(f"User {user}: {recs[:10]}")


Sample recommendations using KNNWithMeans:
User AGTVZ7ZSDMTEDLMJGZRC7RFJNDWQ: ['B0C67HCGBR', 'B0B8M5FJB6', 'B00H35YIJE', 'B079TLFL33', 'B000T9L7W2', 'B007MY5BDI', 'B00H4PEMM6', 'B09YDBKT7M', 'B08R5GM6YB', 'B0002D0Q2W']
User AG44UYFDZHI6FRBQSLDWOGIEOY4A: ['B0C67HCGBR', 'B0B8M5FJB6', 'B000T9L7W2', 'B007MY5BDI', 'B08R5GM6YB', 'B079P9LDHN', 'B0BPKTYTPQ', 'B001W99HE8', 'B000J5UEGQ', 'B01DBS2U9G']
User AE7P3HIBI3UDLCJLUPUZQWYVPEWA: ['B005M0MUQK', 'B0BRS6V8G4', 'B078L11275', 'B09R6KV6QX', 'B07YK57N2M', 'B00RX5HQS4', 'B0BQ4HSKC9', 'B086QM1F75', 'B07R3S93K8', 'B09WF82F1V']
User AE5M7M2VYMM3WHJIBLCHHEU7UOXQ: ['B06XB3FQKB', 'B0BPJ4Q6FJ', 'B0B8M5FJB6', 'B079TLFL33', 'B007MY5BDI', 'B00H4PEMM6', 'B09YDBKT7M', 'B0BXT384GR', 'B079P9LDHN', 'B0BPKH4HB2']
User AESDNHQRRZEWTZF7A7TFFFTSNY5Q: ['B0BGQZNQ53', 'B07L5B64RG', 'B07Y94MSG9', 'B0CB98SMQR', 'B079NS31NK', 'B00CIHB8FY', 'B000SHQ1QC', 'B0BQ4HSKC9', 'B00IZA1GI2', 'B000U0DU34']
Sample recommendations using SVD:
User AGTVZ7ZSDMTEDLMJGZRC7RFJNDWQ: ['B0BPJ4

### 3 Week 8 [Feb 17 - Feb 23]: Evaluation of Recommender Systems

In this session, we will discuss how to evaluate a recommender system. Specifically, let us evaluate all your recommender models from Week 7 on the preprocessed test data split studied in Week 6. You need to:
- Measure the error of the system’s predicted likelihood of rating for the items (Root Mean Square Error, RMSE).
- Discuss the limitations of this metric.

In [59]:
from surprise import accuracy

# Load the preprocessed test data (assuming same structure as train data: user_id, item_id, rating)
test_df = pd.read_csv('Data/clean_test.tsv', sep='\t')

# Convert the test DataFrame to a list of (user, item, rating) tuples
testset = [tuple(x) for x in test_df[['user_id', 'item_id', 'rating']].values]

# Evaluate SVD model on the test set
svd_predictions = svd_best.test(testset)
print("SVD RMSE on test set:")
accuracy.rmse(svd_predictions, verbose=True)

# Evaluate KNNWithMeans model on the test set
knn_predictions = knn_best.test(testset)
print("KNNWithMeans RMSE on test set:")
accuracy.rmse(knn_predictions, verbose=True)

# Note on TopPop:
# The TopPop model is non-personalized and does not output a predicted rating,
# so RMSE is not applicable for this model.

# Discussion of RMSE limitations:
# RMSE measures the average deviation (squared error) between the predicted and actual ratings.
# While it penalizes large errors, RMSE does not consider the ranking of recommendations,
# nor does it capture the utility or relevance of items from a user's perspective.
# In recommender systems, where the goal is to provide a ranked list of items, metrics
# such as precision, recall, or ranking-based measures might provide a more complete picture.

SVD RMSE on test set:
RMSE: 0.9790
KNNWithMeans RMSE on test set:
RMSE: 1.0615


1.0614555427641275

Now, we are interested not in whether the system properly predicts the rating of these items, but rather whether the system gives the best recommendations for each user. To evaluate this, generate the top-k (with k = 10) recommendation for each test user. Based on the top-k recommendation list generated for each user, and using the test data split5, compute:
- Hit rate, averaged across users.
- Precision@k, averaged across users
- Mean Average Precision (MAP@k)
- Mean Reciprocal Rank (MRR@k)
- Coverage

Discuss the advantages and disadvantages of these metrics.
Compare the two types of CF recommender systems and the TopPop system that we have defined so far. Which one works best? Why? What are the advantages and limitations of each approach?

In [62]:
import numpy as np

# Prepare ground truth for each test user (using binary labels)
# Relevant (1) if rating >= 4; not relevant (0) otherwise.
ground_truth = defaultdict(set)
for _, row in test_df.iterrows():
    if row['rating'] >= 4:
        ground_truth[row['user_id']].add(row['item_id'])

# Function to generate top-k recommendations using a given model.
# For personalized models we use the anti-testset from the training set.
def get_top_k(model, trainset, k=10):
    # Build the anti-testset (all unseen items for every user in the trainset)
    anti_testset = trainset.build_anti_testset()
    predictions = model.test(anti_testset)
    
    recommendations = defaultdict(list)
    for pred in predictions:
        recommendations[pred.uid].append((pred.iid, pred.est))
    
    # For each user, sort predicted items in descending order and select top-k.
    for user in recommendations:
        recommendations[user] = [iid for iid, _ in sorted(recommendations[user], key=lambda x: x[1], reverse=True)[:k]]
    return recommendations

# For TopPop (non-personalized), the top-k recommendations are the same for every user.
def get_top_k_for_top_pop(top_pop, test_users, k=10):
    recs = {}
    top_items = top_pop.top_items[:k]
    for user in test_users:
        recs[user] = top_items
    return recs

# Generate recommendations for each model.
# Retrieve the list of test users from ground_truth.
test_users = list(ground_truth.keys())

svd_recs = get_top_k(svd_best, trainset, k=10)
knn_recs = get_top_k(knn_best, trainset, k=10)
top_pop_recs = get_top_k_for_top_pop(top_pop, test_users, k=10)

# Get the total catalog size (we use all items from the training split)
total_items = trainset.n_items

# Evaluation function for hit rate, precision@k, MAP@k, MRR@k and coverage.
def evaluate(recs, ground_truth, k=10, total_items=total_items):
    hit_rates = []
    precisions = []
    avg_precisions = []
    reciprocal_ranks = []
    rec_items = set()
    
    for user, true_items in ground_truth.items():
        recommended = recs.get(user, [])
        rec_items.update(recommended)
        # Hit rate: at least one relevant item recommended.
        hit = 1 if set(recommended) & true_items else 0
        hit_rates.append(hit)
        # Precision@k: fraction of recommended items that are relevant.
        precision = len(set(recommended) & true_items) / k
        precisions.append(precision)
        # Average Precision (AP@k)
        ap = 0
        hit_count = 0
        for i, item in enumerate(recommended):
            if item in true_items:
                hit_count += 1
                ap += hit_count / (i + 1)
        if hit_count > 0:
            ap /= hit_count
        avg_precisions.append(ap)
        # Reciprocal Rank (MRR@k): rank of the first relevant recommendation.
        rr = 0
        for i, item in enumerate(recommended):
            if item in true_items:
                rr = 1 / (i + 1)
                break
        reciprocal_ranks.append(rr)
    
    hit_rate = np.mean(hit_rates)
    precision_at_k = np.mean(precisions)
    map_at_k = np.mean(avg_precisions)
    mrr_at_k = np.mean(reciprocal_ranks)
    # Coverage: unique items recommended over catalog total.
    coverage = len(rec_items) / total_items
    
    return hit_rate, precision_at_k, map_at_k, mrr_at_k, coverage

# Evaluate each recommender.
svd_metrics = evaluate(svd_recs, ground_truth, k=10)
knn_metrics = evaluate(knn_recs, ground_truth, k=10)
top_pop_metrics = evaluate(top_pop_recs, ground_truth, k=10)

print("SVD:    Hit Rate, Precision@10, MAP@10, MRR@10, Coverage =", svd_metrics)
print("KNN:    Hit Rate, Precision@10, MAP@10, MRR@10, Coverage =", knn_metrics)
print("TopPop: Hit Rate, Precision@10, MAP@10, MRR@10, Coverage =", top_pop_metrics)

### Discussion of Metrics
'''
- **Hit Rate:**  
  *Advantages:* Simple to compute and easy to interpret.  
  *Disadvantages:* It is a binary measure that does not reflect how many items in the recommendation list are relevant.

- **Precision@k:**  
  *Advantages:* Measures the proportion of relevant items among the recommended ones.  
  *Disadvantages:* Does not address the ranking order beyond the count.

- **MAP@k:**  
  *Advantages:* Incorporates the ranking order by averaging precision at the points where relevant items appear.  
  *Disadvantages:* More complex and sensitive to the position of the first few relevant items.

- **MRR@k:**  
  *Advantages:* Emphasizes early relevant recommendations (i.e. how quickly a user sees a relevant item).  
  *Disadvantages:* Ignores other relevant items that might appear later in the ranking.

- **Coverage:**  
  *Advantages:* Indicates how diverse the recommendations are across the item catalog.  
  *Disadvantages:* Does not measure relevance, so high coverage does not guarantee user satisfaction.

### Comparison of Recommender Systems

- **Neighborhood-based CF (KNNWithMeans):**
  - *Advantages:* Easy to understand; works well when similar users/items exist.  
  - *Limitations:* Sensitive to sparsity, scaling poorly with many users/items.

- **Latent Factor CF (SVD):**
  - *Advantages:* Captures latent features effectively, often yielding high predictive accuracy.  
  - *Limitations:* Can be more computationally intensive to train; may need careful tuning.

- **TopPop:**
  - *Advantages:* Simple, non-personalized baseline that requires almost no computation.  
  - *Limitations:* Lacks personalization; every user gets the same recommendations, often resulting in lower user satisfaction even if it shows high coverage.

In many practical settings, the SVD-based recommender tends to work best due to its ability to capture the underlying structure in the user-item interaction data. However, the best choice depends on the specific characteristics of the dataset (e.g., sparsity) and the goals (e.g., emphasizing novelty or diversity). The non-personalized TopPop model, while useful as a baseline, does not account for individual user preferences.
'''


SVD:    Hit Rate, Precision@10, MAP@10, MRR@10, Coverage = (0.09466019417475728, 0.010194174757281554, 0.030816574202496536, 0.030776121128062883, 0.1944990176817289)
KNN:    Hit Rate, Precision@10, MAP@10, MRR@10, Coverage = (0.10922330097087378, 0.011650485436893206, 0.030697141316073356, 0.030366774541531822, 0.587426326129666)
TopPop: Hit Rate, Precision@10, MAP@10, MRR@10, Coverage = (0.25728155339805825, 0.032524271844660196, 0.11437432578209276, 0.11868354137771613, 0.019646365422396856)


'\n- **Hit Rate:**  \n  *Advantages:* Simple to compute and easy to interpret.  \n  *Disadvantages:* It is a binary measure that does not reflect how many items in the recommendation list are relevant.\n\n- **Precision@k:**  \n  *Advantages:* Measures the proportion of relevant items among the recommended ones.  \n  *Disadvantages:* Does not address the ranking order beyond the count.\n\n- **MAP@k:**  \n  *Advantages:* Incorporates the ranking order by averaging precision at the points where relevant items appear.  \n  *Disadvantages:* More complex and sensitive to the position of the first few relevant items.\n\n- **MRR@k:**  \n  *Advantages:* Emphasizes early relevant recommendations (i.e. how quickly a user sees a relevant item).  \n  *Disadvantages:* Ignores other relevant items that might appear later in the ranking.\n\n- **Coverage:**  \n  *Advantages:* Indicates how diverse the recommendations are across the item catalog.  \n  *Disadvantages:* Does not measure relevance, so high